---
title: "example with Earthmover"
format: html
---

We will use the monthly regrid data for NE Pacific.
https://app.earthmover.io/NOAA-PMEL/cefi-nep-hindcast-monthly/tree/main/regrid/main

## Using ncdf4

```{r}
library(ncdf4)
```

### Load all the variables 

We don't need to load a specific nc file.

```{r}
url <- "https://compute.earthmover.io/v1/services/dap2/NOAA-PMEL/cefi-nep-hindcast-monthly/main/regrid/main/opendap"
nc <- nc_open(url)
```

Lot's of variables.
```{r}
names(nc$var)
```

### Getting all data for surface

All the data for the surface takes about 3 minutes and is 800+ Mb.

```{r}
tm <- system.time({
  vo_surface_all <- ncvar_get(
    nc,
    varid = "vo_rotate",
    start = c(1, 1, 1, 1),
    count = c(-1, -1, 1, -1)
  )
  })
tm
```

```{r}
dim(vo_surface_all)
format(object.size(vo_surface_all), units = "MB")
```

### Getting data for bounding box

CCE domain, all times, 1 depth

```{r}
lon  <- ncvar_get(nc, "lon")
lat  <- ncvar_get(nc, "lat")

# bounding box
lon_min <- 220
lon_max <- max(lon)
lat_min <- 20
lat_max <- 48

# indices
lon_idx <- which(lon >= lon_min & lon <= lon_max)
lat_idx <- which(lat >= lat_min & lat <= lat_max)

# counts
lon_start <- min(lon_idx)
lon_count <- length(lon_idx)

lat_start <- min(lat_idx)
lat_count <- length(lat_idx)
```

This takes a minute or so.

```{r}
tm <- system.time({
  vo_surface_bbox <- ncvar_get(
    nc,
    varid = "vo_rotate",
    start = c(lon_start, lat_start, 1, 1),
    count = c(lon_count, lat_count, 1, -1)
  )
})
tm
```

```{r}
dim(vo_surface_bbox)
format(object.size(vo_surface_bbox), units = "MB")
```

## Time series of spatial means

```{r}
# coordinates for the subset
lon_sub <- lon[lon_start:(lon_start + lon_count - 1)]
lat_sub <- lat[lat_start:(lat_start + lat_count - 1)]

# time values
time_raw <- ncvar_get(nc, "time")
time_units <- ncatt_get(nc, "time", "units")$value
origin <- sub("days since ", "", time_units)
time_dates <- as.Date(time_raw, origin = origin)

# spatial mean for each time slice
vo_mean_ts <- apply(vo_surface_bbox, 3, mean, na.rm = TRUE)

plot(
  time_dates,
  vo_mean_ts,
  type = "l",
  xlab = "Time",
  ylab = "Mean vo_rotate, surface (m s-1)",
  main = "CCE surface vo_rotate spatial mean"
)
```

### Make a quick plot of first time slice

```{r}
image(
  lon_sub,
  lat_sub,
  vo_surface_bbox[, , 1],
  xlab = "Longitude",
  ylab = "Latitude",
  main = paste("vo_rotate surface", time_dates[1])
)
```

### Make plot with terra

```{r}
library(terra)

m_terra <- t(m)

# terra expects first matrix row to be north/top,
# but lat_sub is south-to-north, so flip matrix rows
m_terra <- m_terra[nrow(m_terra):1, ]

r <- rast(
  m_terra,
  extent = ext(min(lon_sub), max(lon_sub), min(lat_sub), max(lat_sub)),
  crs = "EPSG:4326"
)

plot(r)
```